In [ ]:
FEATURES_DIR.mkdir(parents=True, exist_ok=True)

log.info("=" * 60)
log.info("  FEATURE-ENGINEERING PIPELINE")
log.info("=" * 60)

# ─── Schritt 1: OHLCV-Rohdaten laden ───────────────────────────
log.info("\n" + "─" * 60)
log.info(" SCHRITT 1/3 — OHLCV-Rohdaten laden")
log.info("─" * 60)

def load_ohlcv(files: list[Path]) -> pd.DataFrame:
    log.info(f"  → {len(files)} Parquet-Datei(en): {[f.name for f in files]}")
    df = pd.concat([pd.read_parquet(f) for f in files], ignore_index=True)
    log.info(f"  → Nach concat:  Shape {df.shape}")
    print(df.head(3).to_string(index=False))

    df["timestamp"] = pd.to_datetime(df["timestamp"], unit="ms", utc=True)
    df = df.sort_values(["market_symbol", "timestamp"]).reset_index(drop=True)
    log.info(f"\n  → Nach Timestamp-Konvertierung & Sortierung:  Shape {df.shape}")
    print(df[["timestamp", "market_symbol", "open", "close", "volume_quote"]].head(5).to_string(index=False))
    return df

files  = sorted(ASSET_OHLCV_5M_DIR.glob("ohlcv_*.parquet"))
df_raw = load_ohlcv(files)

log.info(f"\n  Zusammenfassung df_raw:")
log.info(f"    Kerzen:   {len(df_raw):,}")
log.info(f"    Ticker:   {df_raw['market_symbol'].nunique()}")
log.info(f"    Tage:     {df_raw['timestamp'].dt.date.nunique()}")
log.info(f"    Zeitraum: {df_raw['timestamp'].min()} → {df_raw['timestamp'].max()}")


# ── RSI ──────────────────────────────────────────────────────
def calc_rsi(close: pd.Series, period: int = 14) -> pd.Series:
    """
    RSI nach Wilder.
    Werte 0–100. Unter 30 = überverkauft, über 70 = überkauft.
    """
    delta    = close.diff()
    gain     = delta.clip(lower=0)
    loss     = (-delta).clip(lower=0)
    avg_gain = gain.ewm(alpha=1 / period, min_periods=period, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1 / period, min_periods=period, adjust=False).mean()
    rs       = avg_gain / avg_loss.replace(0, np.nan)
    return (100 - (100 / (1 + rs))).rename(f"rsi_{period}")


# ── Bollinger Bands ───────────────────────────────────────────
def calc_bollinger(series: pd.Series, period: int = 20, num_std: float = 2.0) -> pd.DataFrame:
    """
    bb_pct = Position innerhalb der Bänder (0=unten, 1=oben).
    """
    mid   = series.rolling(period).mean()
    std   = series.rolling(period).std()
    upper = mid + num_std * std
    lower = mid - num_std * std
    pct   = (series - lower) / (upper - lower).replace(0, np.nan)
    return pd.DataFrame({
        f"bb_mid_{period}":   mid,
        f"bb_upper_{period}": upper,
        f"bb_lower_{period}": lower,
        f"bb_pct_{period}":   pct,
    })


# ── ATR ───────────────────────────────────────────────────────
def calc_atr(high: pd.Series, low: pd.Series, close: pd.Series,
             period: int = 14) -> pd.Series:
    """
    Average True Range nach Wilder.
    """
    prev_close = close.shift(1)
    tr = pd.concat([
        high - low,
        (high - prev_close).abs(),
        (low  - prev_close).abs(),
    ], axis=1).max(axis=1)
    return tr.ewm(alpha=1 / period, min_periods=period, adjust=False).mean().rename(f"atr_{period}")


# ── VWAP ──────────────────────────────────────────────────────
def calc_vwap(high: pd.Series, low: pd.Series, close: pd.Series,
              volume: pd.Series, timestamp: pd.Series) -> pd.Series:
    """
    Intraday VWAP — wird täglich zurückgesetzt.
    """
    typical_price = (high + low + close) / 3
    date          = timestamp.dt.date
    cum_tp_vol    = (typical_price * volume).groupby(date).cumsum()
    cum_vol       = volume.groupby(date).cumsum()
    return (cum_tp_vol / cum_vol.replace(0, np.nan)).rename("vwap")


# ── Z-Score (rollierend) ──────────────────────────────────────
def calc_zscore(series: pd.Series, period: int = 20) -> pd.Series:
    """
    Rollierender Z-Score. |z| > 2 = potenzielles Einstiegssignal.
    """
    mu  = series.rolling(period).mean()
    std = series.rolling(period).std()
    return ((series - mu) / std.replace(0, np.nan)).rename(f"zscore_{period}")


# ── Hurst-Exponent (rollierend) ───────────────────────────────
def calc_hurst(series: pd.Series, window: int = 100,
               min_lag: int = 2, max_lag: int = 20) -> pd.Series:
    """
    H < 0.5 → mean-reverting  |  H ≈ 0.5 → random walk  |  H > 0.5 → trending
    """
    def hurst_single(x: np.ndarray) -> float:
        lags = range(min_lag, max_lag + 1)
        tau  = [np.std(np.subtract(x[lag:], x[:-lag])) for lag in lags]
        if any(t == 0 for t in tau):
            return np.nan
        return np.polyfit(np.log(lags), np.log(tau), 1)[0]
    return series.rolling(window).apply(hurst_single, raw=True).rename(f"hurst_{window}")


# ── Half-Life der Mean Reversion ──────────────────────────────
def calc_halflife(series: pd.Series, window: int = 100) -> pd.Series:
    """
    Ornstein-Uhlenbeck Half-Life (in Perioden).
    NaN = kein Mean-Reversion-Regime in diesem Fenster.
    """
    def halflife_single(x: np.ndarray) -> float:
        delta = x[1:] - x[:-1]
        y_lag = x[:-1]
        var   = np.var(y_lag)
        if var == 0:
            return np.nan
        lam = np.cov(y_lag, delta)[0, 1] / var
        if lam >= 0:
            return np.nan
        return -np.log(2) / lam
    return series.rolling(window).apply(halflife_single, raw=True).rename(f"halflife_{window}")


# ─── Schritt 2: Indikatoren berechnen ──────────────────────────
log.info("\n" + "─" * 60)
log.info(" SCHRITT 2/3 — Indikatoren berechnen")
log.info("─" * 60)
log.info(f"  Verarbeite {df_raw['market_symbol'].nunique()} Ticker …")

def compute_indicators(df: pd.DataFrame) -> pd.DataFrame:
    results      = []
    total        = df["market_symbol"].nunique()
    first_ticker = True

    for i, (ticker, group) in enumerate(df.groupby("market_symbol", sort=False), 1):
        g = group.copy().sort_values("timestamp").reset_index(drop=True)

        g["rsi_7"]        = calc_rsi(g["close"], period=7).values
        g["rsi_14"]       = calc_rsi(g["close"], period=14).values
        g["atr_14"]       = calc_atr(g["high"], g["low"], g["close"], period=14).values
        g["vwap"]         = calc_vwap(g["high"], g["low"], g["close"],
                                      g["volume_quote"], g["timestamp"]).values
        g["vwap_dev"]     = (g["close"] - g["vwap"]) / g["vwap"]
        bb                = calc_bollinger(g["close"], period=20)
        g                 = pd.concat([g, bb], axis=1)
        g["zscore_20"]    = calc_zscore(g["close"], period=20).values
        g["zscore_60"]    = calc_zscore(g["close"], period=60).values
        g["hurst_100"]    = calc_hurst(g["close"], window=100).values
        g["halflife_100"] = calc_halflife(g["close"], window=100).values

        results.append(g)
        nan_rsi = g["rsi_14"].isna().sum()
        log.info(f"  [{i:>3}/{total}] {ticker}: {len(g)} Kerzen  |  NaN rsi_14: {nan_rsi}")

        if first_ticker:
            first_ticker = False
            ind_cols = ["timestamp", "close", "rsi_14", "bb_pct_20",
                        "zscore_20", "vwap_dev", "atr_14", "hurst_100", "halflife_100"]
            log.info(f"\n  Vorschau Indikatoren — {ticker} (erste 5 Zeilen):")
            log.info(g[ind_cols].head(5).to_string(index=False))
            print()

    df_out = pd.concat(results, ignore_index=True)
    return df_out

df_features = compute_indicators(df_raw)

log.info(f"\n  → df_features Shape: {df_features.shape}")
log.info(f"  → Spalten ({len(df_features.columns)}): {list(df_features.columns)}")
feat_cols = ["timestamp", "market_symbol", "close", "rsi_14", "bb_pct_20",
             "zscore_20", "vwap_dev", "atr_14", "hurst_100", "halflife_100"]
log.info(f"\n  Vorschau df_features (erste 5 Zeilen):")
print(df_features[feat_cols].head(5).to_string(index=False))


# ─── Schritt 3: Speichern & Schnellcheck ───────────────────────
log.info("\n" + "─" * 60)
log.info(" SCHRITT 3/3 — Speichern & Schnellcheck")
log.info("─" * 60)

out = FEATURES_DIR / "ohlcv_features.parquet"
df_features.to_parquet(out, index=False)
log.info(f"  → Gespeichert: {out}")
log.info(f"  → Dateigrösse: {out.stat().st_size / 1_048_576:.1f} MB")
log.info(f"  → Zeilen: {len(df_features):,}  |  Spalten: {len(df_features.columns)}")

SAMPLE_TICKER = "BTCUSDT"
sample = df_features[df_features["market_symbol"] == SAMPLE_TICKER].tail(20)
cols = ["timestamp", "close", "rsi_14", "bb_pct_20",
        "zscore_20", "vwap_dev", "atr_14", "hurst_100", "halflife_100"]

display_df = sample[cols].copy()
display_df["timestamp"]    = display_df["timestamp"].dt.strftime("%Y-%m-%d %H:%M")
display_df["close"]        = display_df["close"].map("${:,.2f}".format)
display_df["rsi_14"]       = display_df["rsi_14"].map("{:.1f}".format)
display_df["bb_pct_20"]    = display_df["bb_pct_20"].map("{:.3f}".format)
display_df["zscore_20"]    = display_df["zscore_20"].map("{:.3f}".format)
display_df["vwap_dev"]     = display_df["vwap_dev"].map("{:.4f}".format)
display_df["atr_14"]       = display_df["atr_14"].map("{:.2f}".format)
display_df["hurst_100"]    = display_df["hurst_100"].map(lambda x: f"{x:.3f}" if pd.notna(x) else "–")
display_df["halflife_100"] = display_df["halflife_100"].map(lambda x: f"{x:.1f}" if pd.notna(x) else "–")

log.info(f"\n  {SAMPLE_TICKER} — letzte 20 Kerzen:")
print(display_df.to_string(index=False))

log.info("\n" + "=" * 60)
log.info("  FEATURE-ENGINEERING ABGESCHLOSSEN")
log.info("=" * 60)


# Feature-Engineering Pipeline — Ausführliche Code-Erklärung

---

## Gesamtüberblick

**Was es tut (in einem Satz):**
Es lädt rohe 5-Minuten-Kerzendaten (OHLCV), berechnet 7 technische Indikatoren pro Kryptowährung
und speichert alles als eine einzige Parquet-Datei für nachgelagerte Modelle oder Backtests.

**Datenfluss:**

```
Parquet-Dateien (roh)
        │
        ▼
   load_ohlcv()          ← Dateien zusammenführen, Timestamps konvertieren
        │
        ▼
compute_indicators()     ← Pro Ticker alle Indikatoren berechnen
   ├── calc_rsi()             RSI (7 und 14)
   ├── calc_atr()             Average True Range
   ├── calc_vwap()            VWAP + Abweichung
   ├── calc_bollinger()       Bollinger Bands
   ├── calc_zscore()          Z-Score (20 und 60)
   ├── calc_hurst()           Hurst-Exponent
   └── calc_halflife()        Half-Life
        │
        ▼
  ohlcv_features.parquet  ← Fertiges Feature-Set
```

---

## 1. Verzeichnis anlegen

```python
FEATURES_DIR = PROJECT_ROOT / "data" / "processed" / "features"
FEATURES_DIR.mkdir(parents=True, exist_ok=True)
```

**Was passiert:**
- Definiert den Ausgabeordner für die fertigen Features.
- `mkdir(parents=True, exist_ok=True)` erstellt den gesamten Pfad inklusive
  aller Eltern-Ordner. Wenn der Ordner schon existiert, wird kein Fehler geworfen.
- `PROJECT_ROOT / "data" / ...` nutzt Pythons `pathlib`-Syntax — der `/`-Operator
  verkettet Pfad-Segmente plattformunabhängig.

---

## 2. OHLCV-Daten laden

```python
def load_ohlcv(files: list[Path]) -> pd.DataFrame:
    df = pd.concat([pd.read_parquet(f) for f in files], ignore_index=True)
    df["timestamp"] = pd.to_datetime(df["timestamp"], unit="ms", utc=True)
    df = df.sort_values(["market_symbol", "timestamp"]).reset_index(drop=True)
    return df

files  = sorted(ASSET_OHLCV_5M_DIR.glob("ohlcv_*.parquet"))
df_raw = load_ohlcv(files)
print(f"Geladen: {len(df_raw):,} Kerzen | "
      f"{df_raw['market_symbol'].nunique()} Ticker | "
      f"{df_raw['timestamp'].dt.date.nunique()} Tage")
```

### Zeile für Zeile:

**`pd.concat([pd.read_parquet(f) for f in files], ignore_index=True)`**
- List Comprehension: Liest jede Parquet-Datei einzeln in einen DataFrame.
- `pd.concat(...)` fügt alle DataFrames vertikal zusammen (untereinander).
- `ignore_index=True` setzt den Index neu durch (0, 1, 2, ...) statt
  die alten Indizes zu behalten, die sich sonst überlappen würden.

**`pd.to_datetime(df["timestamp"], unit="ms", utc=True)`**
- Die Rohdaten enthalten Timestamps als Ganzzahlen (Millisekunden seit 1.1.1970).
- `unit="ms"` sagt Pandas: interpretiere die Zahlen als Millisekunden.
- `utc=True` macht die Zeitzone explizit UTC — wichtig für korrekte Tagesgrenz-Berechnungen (VWAP).
- Beispiel: `1717200000000` → `2024-06-01 00:00:00+00:00`

**`df.sort_values(["market_symbol", "timestamp"])`**
- Sortiert zuerst nach Ticker (z.B. BTCUSDT, ETHUSDT), dann innerhalb jedes Tickers nach Zeit.
- Das ist **zwingend nötig**, weil alle nachfolgenden Indikatoren auf chronologischer
  Reihenfolge basieren (`.diff()`, `.shift()`, `.rolling()`, `.ewm()` etc.).

**`sorted(ASSET_OHLCV_5M_DIR.glob("ohlcv_*.parquet"))`**
- `glob("ohlcv_*.parquet")` findet alle Dateien im Verzeichnis, deren Name mit
  `ohlcv_` beginnt und auf `.parquet` endet.
- `sorted()` sortiert alphabetisch/chronologisch — z.B. `ohlcv_2024_06.parquet`
  kommt vor `ohlcv_2024_07.parquet`.

**Die `print`-Zeile:**
- Gibt eine Zusammenfassung aus, z.B.: `Geladen: 2,340,000 Kerzen | 50 Ticker | 30 Tage`
- `len(df_raw):,` → Zahl mit Tausender-Trennzeichen.
- `.nunique()` → Anzahl einzigartiger Werte.

---

## 3. RSI (Relative Strength Index)

```python
def calc_rsi(close: pd.Series, period: int = 14) -> pd.Series:
    delta    = close.diff()
    gain     = delta.clip(lower=0)
    loss     = (-delta).clip(lower=0)
    avg_gain = gain.ewm(alpha=1 / period, min_periods=period, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1 / period, min_periods=period, adjust=False).mean()
    rs       = avg_gain / avg_loss.replace(0, np.nan)
    return (100 - (100 / (1 + rs))).rename(f"rsi_{period}")
```

### Was der RSI misst:
Die **Stärke der Kursbewegung** auf einer Skala von 0 bis 100.
Er vergleicht: Wie stark sind die Aufwärtsbewegungen im Vergleich zu den Abwärtsbewegungen?

### Zeile für Zeile:

**`delta = close.diff()`**
- Berechnet die Differenz jedes Wertes zum Vorgänger.
- Beispiel: close = [100, 102, 101, 104] → delta = [NaN, +2, -1, +3]
- Der erste Wert ist immer NaN, weil es keinen Vorgänger gibt.

**`gain = delta.clip(lower=0)`**
- `clip(lower=0)` setzt alle negativen Werte auf 0, behält positive bei.
- Ergebnis: Nur die **Gewinne** (positive Preisänderungen).
- Beispiel: delta = [NaN, +2, -1, +3] → gain = [NaN, 2, 0, 3]

**`loss = (-delta).clip(lower=0)`**
- Zuerst wird delta negiert: [NaN, -2, +1, -3]
- Dann wieder clip(lower=0): Nur die **Verluste** (als positive Zahlen).
- Beispiel: → loss = [NaN, 0, 1, 0]

**`avg_gain = gain.ewm(alpha=1/period, min_periods=period, adjust=False).mean()`**
- `ewm()` = Exponentially Weighted Moving average.
- `alpha=1/period` → Bei period=14 ist alpha=1/14 ≈ 0.071.
  Das bedeutet: Jeder neue Wert bekommt ~7.1% Gewicht, der Rest verteilt
  sich exponentiell abnehmend auf ältere Werte.
- `min_periods=period` → Die ersten 13 Werte geben NaN zurück.
  Erst ab dem 14. Wert gibt es ein Ergebnis.
- `adjust=False` → Verwendet die rekursive Formel:
  `avg_gain[t] = alpha × gain[t] + (1 - alpha) × avg_gain[t-1]`
  Das ist exakt Wilder's Original-Methode. Mit `adjust=True` (Standard)
  würde Pandas eine andere Gewichtung verwenden.

**`avg_loss = loss.ewm(...).mean()`**
- Identisch, aber für die Verlustserie.

**`rs = avg_gain / avg_loss.replace(0, np.nan)`**
- RS = Relative Strength = Durchschnittlicher Gewinn / Durchschnittlicher Verlust.
- `.replace(0, np.nan)` verhindert Division durch 0.
  Wenn es keine Verluste gab, ist avg_loss=0 → Division wäre undefiniert.
- Wenn RS = 1 → gleich viel Gewinn wie Verlust.
- Wenn RS > 1 → mehr Gewinn als Verlust.
- Wenn RS < 1 → mehr Verlust als Gewinn.

**`return (100 - (100 / (1 + rs))).rename(f"rsi_{period}")`**
- Transformiert RS auf die Skala 0–100:
  - RS = 0 → RSI = 0 (nur Verluste)
  - RS = 1 → RSI = 50 (ausgeglichen)
  - RS → ∞ → RSI → 100 (nur Gewinne)
- `.rename(f"rsi_{period}")` benennt die Series, z.B. `"rsi_14"`.

### Interpretation:
| RSI-Wert | Bedeutung |
|---|---|
| > 70 | Überkauft — möglicherweise zu stark gestiegen |
| 30–70 | Neutral |
| < 30 | Überverkauft — möglicherweise zu stark gefallen |

---

## 4. Bollinger Bands

```python
def calc_bollinger(series: pd.Series, period: int = 20, num_std: float = 2.0) -> pd.DataFrame:
    mid   = series.rolling(period).mean()
    std   = series.rolling(period).std()
    upper = mid + num_std * std
    lower = mid - num_std * std
    pct   = (series - lower) / (upper - lower).replace(0, np.nan)
    return pd.DataFrame({
        f"bb_mid_{period}":   mid,
        f"bb_upper_{period}": upper,
        f"bb_lower_{period}": lower,
        f"bb_pct_{period}":   pct,
    })
```

### Was Bollinger Bands messen:
Einen **dynamischen Volatilitätskanal** um den Durchschnittspreis.
Die Bänder werden breiter bei hoher Volatilität und enger bei niedriger.

### Zeile für Zeile:

**`mid = series.rolling(period).mean()`**
- Gleitender Durchschnitt der letzten 20 Schlusskurse.
- `rolling(20)` erstellt ein "Fenster" von 20 Werten, das sich durch die Daten schiebt.
- `.mean()` berechnet den Durchschnitt innerhalb jedes Fensters.
- Die ersten 19 Werte sind NaN (Fenster noch nicht voll).

**`std = series.rolling(period).std()`**
- Standardabweichung der letzten 20 Werte.
- Misst die Streuung/Volatilität innerhalb des Fensters.

**`upper = mid + num_std * std`**
- Oberes Band = Durchschnitt + 2 × Standardabweichung.
- Statistisch: Bei Normalverteilung liegen ~95% der Werte innerhalb von ±2 Std.abw.

**`lower = mid - num_std * std`**
- Unteres Band = Durchschnitt − 2 × Standardabweichung.

**`pct = (series - lower) / (upper - lower).replace(0, np.nan)`**
- Die **relative Position** des aktuellen Preises innerhalb der Bänder.
- Formel: Wo steht der Preis zwischen unterem und oberem Band?
- `.replace(0, np.nan)` → Wenn oberes und unteres Band identisch sind
  (Volatilität = 0), wird Division durch 0 verhindert.
- Ergebnis:
  - pct = 0.0 → Preis liegt genau auf dem unteren Band
  - pct = 0.5 → Preis liegt genau in der Mitte
  - pct = 1.0 → Preis liegt genau auf dem oberen Band
  - pct > 1.0 → Preis liegt **über** dem oberen Band (extrem)
  - pct < 0.0 → Preis liegt **unter** dem unteren Band (extrem)

**`return pd.DataFrame({...})`**
- Gibt einen DataFrame mit 4 Spalten zurück:
  `bb_mid_20`, `bb_upper_20`, `bb_lower_20`, `bb_pct_20`.

### Visuell:

```
Preis
  │
  │   ╭──── bb_upper_20 (Durchschnitt + 2σ)
  │   │
  │   │  ── bb_mid_20   (20er Durchschnitt)
  │   │
  │   ╰──── bb_lower_20 (Durchschnitt − 2σ)
  │
  └──────────────────────── Zeit
```

---

## 5. ATR (Average True Range)

```python
def calc_atr(high: pd.Series, low: pd.Series, close: pd.Series,
             period: int = 14) -> pd.Series:
    prev_close = close.shift(1)
    tr = pd.concat([
        high - low,
        (high - prev_close).abs(),
        (low  - prev_close).abs(),
    ], axis=1).max(axis=1)
    return tr.ewm(alpha=1 / period, min_periods=period, adjust=False).mean().rename(f"atr_{period}")
```

### Was die ATR misst:
Die **durchschnittliche Schwankungsbreite** pro Kerze — also wie viel sich der Preis
typischerweise innerhalb einer 5-Minuten-Kerze bewegt.

### Zeile für Zeile:

**`prev_close = close.shift(1)`**
- Verschiebt die Close-Serie um eine Position nach unten.
- Jede Zeile enthält jetzt den **Schlusskurs der vorherigen Kerze**.
- Beispiel: close = [100, 102, 101] → prev_close = [NaN, 100, 102]

**`high - low`**
- Die Intraday-Spanne: Wie weit lagen High und Low innerhalb der Kerze auseinander.
- Das ist die "offensichtliche" Volatilität.

**`(high - prev_close).abs()`**
- Abstand zwischen dem heutigen Hoch und dem gestrigen Schluss.
- Fängt **Aufwärts-Gaps** ein: Wenn der Markt mit einer Lücke nach oben öffnet,
  ist high - low möglicherweise klein, aber die tatsächliche Bewegung war gross.

**`(low - prev_close).abs()`**
- Abstand zwischen dem heutigen Tief und dem gestrigen Schluss.
- Fängt **Abwärts-Gaps** ein.

**`pd.concat([...], axis=1).max(axis=1)`**
- Setzt alle drei Berechnungen nebeneinander (axis=1 = Spalten).
- `.max(axis=1)` nimmt pro Zeile den **grössten** der drei Werte.
- Das ist der **True Range** — er berücksichtigt immer den "schlimmsten Fall".

**Beispiel für Gap:**
```
Vorherige Kerze: Close = 100
Aktuelle Kerze:  High = 110, Low = 108, Close = 109

high - low       = 110 - 108 =  2   (sieht harmlos aus)
high - prev_close = 110 - 100 = 10   (← DAS ist die wahre Bewegung!)
low  - prev_close = 108 - 100 =  8

True Range = max(2, 10, 8) = 10
```

**`tr.ewm(alpha=1/period, min_periods=period, adjust=False).mean()`**
- Glättet den True Range mit Wilder's exponentieller Methode (identisch wie beim RSI).
- Ergebnis: Ein stabiler Wert, der nicht von einzelnen Ausreissern dominiert wird.

### Verwendung in der Strategie:
- **Stop-Loss**: z.B. Stop = Einstiegspreis − 2 × ATR
- **Positionsgrösse**: Höhere ATR → kleinere Position (gleich viel Risiko pro Trade)

---

## 6. VWAP (Volume Weighted Average Price)

```python
def calc_vwap(high: pd.Series, low: pd.Series, close: pd.Series,
              volume: pd.Series, timestamp: pd.Series) -> pd.Series:
    typical_price = (high + low + close) / 3
    date          = timestamp.dt.date
    cum_tp_vol    = (typical_price * volume).groupby(date).cumsum()
    cum_vol       = volume.groupby(date).cumsum()
    return (cum_tp_vol / cum_vol.replace(0, np.nan)).rename("vwap")
```

### Was der VWAP misst:
Den **volumengewichteten Durchschnittspreis** seit Tagesbeginn.
Er zeigt, zu welchem Preis die meisten Transaktionen (nach Volumen) stattfanden.

### Zeile für Zeile:

**`typical_price = (high + low + close) / 3`**
- Der "typische Preis" pro Kerze — ein Kompromiss zwischen High, Low und Close.
- Fairere Repräsentation als nur Close, weil er das Intraday-Verhalten berücksichtigt.

**`date = timestamp.dt.date`**
- Extrahiert nur das Datum (ohne Uhrzeit) aus dem Timestamp.
- Wird als Gruppierungsschlüssel für den täglichen Reset verwendet.

**`cum_tp_vol = (typical_price * volume).groupby(date).cumsum()`**
- `typical_price * volume` = Gewichteter Preis-Beitrag jeder Kerze.
- `.groupby(date)` → Gruppiert nach Tag, damit die Kumulierung täglich neu beginnt.
- `.cumsum()` → Kumulierte (laufende) Summe innerhalb jedes Tages.
- Ergebnis: Am Ende des Tages steht hier die Summe aller (Preis × Volumen) des ganzen Tages.

**`cum_vol = volume.groupby(date).cumsum()`**
- Kumuliertes Volumen seit Tagesbeginn.

**`cum_tp_vol / cum_vol.replace(0, np.nan)`**
- VWAP = Kumulierter (Preis × Volumen) / Kumuliertes Volumen.
- `.replace(0, np.nan)` → Wenn noch kein Volumen gehandelt wurde, NaN statt Division durch 0.

### Warum `groupby(date)` wichtig ist:
Ohne groupby würde die kumulative Summe **über Tagesgrenzen hinweg** weiterlaufen.
Der VWAP ist aber ein **Intraday-Indikator** — er wird jeden Tag um Mitternacht
auf 0 zurückgesetzt und beginnt neu.

### Im Hauptcode wird zusätzlich berechnet:

```python
g["vwap_dev"] = (g["close"] - g["vwap"]) / g["vwap"]
```

- Die **relative Abweichung** vom VWAP in Prozent.
- Positiv → Preis liegt über dem volumengewichteten Durchschnitt (bullish).
- Negativ → Preis liegt darunter (bearish).
- Beispiel: close=101, vwap=100 → vwap_dev = 0.01 = 1% über VWAP.

---

## 7. Z-Score (rollierend)

```python
def calc_zscore(series: pd.Series, period: int = 20) -> pd.Series:
    mu  = series.rolling(period).mean()
    std = series.rolling(period).std()
    return ((series - mu) / std.replace(0, np.nan)).rename(f"zscore_{period}")
```

### Was der Z-Score misst:
**Wie viele Standardabweichungen** der aktuelle Preis vom rollierenden
Durchschnitt entfernt ist. Das ist das **primäre Einstiegssignal** der Strategie.

### Zeile für Zeile:

**`mu = series.rolling(period).mean()`**
- Rollierender Durchschnitt der letzten 20 (bzw. 60) Schlusskurse.
- Das ist der "erwartete" Wert — der Mittelwert, zu dem der Preis
  bei Mean-Reversion zurückkehren sollte.

**`std = series.rolling(period).std()`**
- Rollierende Standardabweichung — das Mass für "normale" Schwankungen.

**`(series - mu) / std`**
- Die Z-Score-Formel: (aktueller Wert − Durchschnitt) / Standardabweichung.
- Normiert die Abweichung: Egal ob BTC bei $60'000 oder ein Altcoin bei $0.50 steht,
  ein Z-Score von 2 bedeutet immer "2 Standardabweichungen über dem Durchschnitt".

### Interpretation:

| Z-Score | Bedeutung | Aktion (Mean-Reversion) |
|---|---|---|
| +3 oder mehr | Extrem über Durchschnitt | Starkes Short-Signal |
| +2 | Deutlich über Durchschnitt | Short-Signal |
| 0 | Genau beim Durchschnitt | Kein Signal |
| −2 | Deutlich unter Durchschnitt | Long-Signal |
| −3 oder weniger | Extrem unter Durchschnitt | Starkes Long-Signal |

### Warum zwei Perioden?
- `zscore_20` → Kurzfristig (20 × 5min = 100 Minuten Rückblick)
- `zscore_60` → Mittelfristig (60 × 5min = 5 Stunden Rückblick)
- Wenn **beide** Z-Scores extrem sind, ist das Signal zuverlässiger.

---

## 8. Hurst-Exponent (rollierend)

```python
def calc_hurst(series: pd.Series, window: int = 100,
               min_lag: int = 2, max_lag: int = 20) -> pd.Series:
    def hurst_single(x: np.ndarray) -> float:
        lags = range(min_lag, max_lag + 1)
        tau  = [np.std(np.subtract(x[lag:], x[:-lag])) for lag in lags]
        if any(t == 0 for t in tau):
            return np.nan
        return np.polyfit(np.log(lags), np.log(tau), 1)[0]

    return series.rolling(window).apply(hurst_single, raw=True).rename(f"hurst_{window}")
```

### Was der Hurst-Exponent misst:
Ob die Preisserie aktuell **mean-reverting**, **zufällig** oder **trendend** ist.
Das ist ein **Regime-Filter** — er entscheidet, ob die Strategie überhaupt handeln sollte.

### Die innere Funktion `hurst_single(x)` im Detail:

**`lags = range(min_lag, max_lag + 1)`**
- Erstellt die Lag-Werte [2, 3, 4, ..., 20].
- Ein "Lag" bedeutet: Wie weit schaut man zurück für den Vergleich?

**`tau = [np.std(np.subtract(x[lag:], x[:-lag])) for lag in lags]`**
- Für jeden Lag-Wert wird berechnet:
  1. `x[lag:]` = Alle Werte ab Position `lag` (die "späteren" Werte).
  2. `x[:-lag]` = Alle Werte bis Position `-lag` (die "früheren" Werte).
  3. `np.subtract(...)` = Differenz zwischen späteren und früheren Werten.
  4. `np.std(...)` = Standardabweichung dieser Differenzen.
- `tau[lag]` misst also: **Wie stark streuen die Preisänderungen über `lag` Perioden?**
- Beispiel mit lag=5: Wie unterschiedlich sind Preise, die 5 Kerzen auseinanderliegen?

**`if any(t == 0 for t in tau): return np.nan`**
- Sicherheitscheck: Wenn für irgendeinen Lag die Streuung 0 ist
  (z.B. bei konstantem Preis), kann man den Hurst-Exponent nicht berechnen.

**`return np.polyfit(np.log(lags), np.log(tau), 1)[0]`**
- Lineare Regression im **log-log-Raum**:
  - X-Achse: ln(lag) → [ln(2), ln(3), ..., ln(20)]
  - Y-Achse: ln(tau) → die logarithmierten Streuungswerte
- `np.polyfit(..., 1)` fittet eine Gerade (Polynom 1. Grades).
- `[0]` nimmt die **Steigung** dieser Geraden — das ist der Hurst-Exponent H.

**Warum log-log?**
Für eine Zeitreihe gilt theoretisch: `tau ∝ lag^H`
Logarithmiert: `ln(tau) = H × ln(lag) + const`
→ Die Steigung im log-log-Plot ist direkt der Hurst-Exponent.

### `series.rolling(window).apply(hurst_single, raw=True)`
- `rolling(100)` → Fenster von 100 Kerzen (≈ 8.3 Stunden bei 5min).
- `.apply(hurst_single, raw=True)` → Wendet die Funktion auf jedes Fenster an.
- `raw=True` → Übergibt ein NumPy-Array statt einer Pandas-Series (schneller).

### Interpretation:

| H-Wert | Verhalten | Für die Strategie |
|---|---|---|
| H < 0.5 | Mean-reverting: Preis kehrt zum Mittelwert zurück | ✅ Handeln! |
| H ≈ 0.5 | Random Walk: Keine Vorhersagbarkeit | ⚠️ Vorsicht |
| H > 0.5 | Trending: Preis bewegt sich weiter in eine Richtung | ❌ Nicht handeln |

### Hinweis zur Performance:
Diese Berechnung ist **rechenintensiv**: Für jede der 100er-Fenster-Positionen
wird eine Regression durchgeführt. Bei Millionen von Kerzen kann das Minuten dauern.

---

## 9. Half-Life der Mean Reversion

```python
def calc_halflife(series: pd.Series, window: int = 100) -> pd.Series:
    def halflife_single(x: np.ndarray) -> float:
        delta = x[1:] - x[:-1]
        y_lag = x[:-1]
        var   = np.var(y_lag)
        if var == 0:
            return np.nan
        lam = np.cov(y_lag, delta)[0, 1] / var
        if lam >= 0:
            return np.nan
        return -np.log(2) / lam

    return series.rolling(window).apply(halflife_single, raw=True).rename(f"halflife_{window}")
```

### Was die Half-Life misst:
**Wie viele Perioden** es dauert, bis eine Abweichung vom Mittelwert
um die Hälfte abgebaut ist. Das bestimmt den **Time-Stop** der Strategie.

### Die innere Funktion `halflife_single(x)` im Detail:

**Theoretischer Hintergrund: Ornstein-Uhlenbeck-Prozess**

Das Modell nimmt an, dass der Preis einem Mean-Reversion-Prozess folgt:
```
Δx(t) = λ · x(t-1) + ε(t)
```
Wenn λ negativ ist, wird der Preis zum Mittelwert zurückgezogen.

**`delta = x[1:] - x[:-1]`**
- Die Preisänderungen: Jeder Wert minus sein Vorgänger.
- `x[1:]` = [x₁, x₂, ..., x₉₉]
- `x[:-1]` = [x₀, x₁, ..., x₉₈]
- Ergebnis: [x₁−x₀, x₂−x₁, ..., x₉₉−x₉₈] — das sind die Δx-Werte.

**`y_lag = x[:-1]`**
- Die vorherigen Preisniveaus — die "erklärende Variable" in der Regression.

**`var = np.var(y_lag)`**
- Varianz der Preisniveaus.
- Wenn var=0 → Der Preis war konstant → Keine Mean-Reversion berechenbar.

**`lam = np.cov(y_lag, delta)[0, 1] / var`**
- Das ist eine **OLS-Regression** (Ordinary Least Squares) ohne Intercept.
- `np.cov(y_lag, delta)` gibt die 2×2 Kovarianzmatrix zurück:
  ```
  [[Var(y_lag),      Cov(y_lag, delta)],
   [Cov(y_lag, delta), Var(delta)      ]]
  ```
- `[0, 1]` extrahiert die **Kovarianz** zwischen y_lag und delta.
- `Cov / Var` = Regressionskoeffizient β = λ.
- λ ist die **Mean-Reversion-Geschwindigkeit**.

**`if lam >= 0: return np.nan`**
- Wenn λ ≥ 0, gibt es **keine Mean-Reversion** in diesem Fenster.
- λ > 0 würde bedeuten: Je höher der Preis, desto stärker steigt er weiter (Trend).
- In dem Fall gibt die Funktion NaN zurück — "keine Information".

**`return -np.log(2) / lam`**
- Die Half-Life-Formel: t½ = −ln(2) / λ
- Beispiel: λ = −0.023 → t½ = −0.693 / (−0.023) = 30 Perioden.
- Bei 5-Minuten-Kerzen: 30 × 5 = 150 Minuten = 2.5 Stunden.
- Bedeutung: Eine Abweichung vom Mittelwert wird in ca. 2.5 Stunden
  zur Hälfte abgebaut.

### Verwendung:
- **Time-Stop**: Wenn ein Trade nach `halflife` Perioden nicht profitabel ist,
  wird er geschlossen — die erwartete Mean-Reversion ist offensichtlich nicht eingetreten.
- Kurze Half-Life (z.B. 10–30) → Schnelle Rückkehr → Aggressiver handeln.
- Lange Half-Life (z.B. 100+) → Langsame Rückkehr → Konservativer sein.

---

## 10. Alle Indikatoren zusammenführen

```python
def compute_indicators(df: pd.DataFrame) -> pd.DataFrame:
    results = []
    for ticker, group in df.groupby("market_symbol", sort=False):
        g = group.copy().sort_values("timestamp").reset_index(drop=True)

        g["rsi_7"]    = calc_rsi(g["close"], period=7).values
        g["rsi_14"]   = calc_rsi(g["close"], period=14).values
        g["atr_14"]   = calc_atr(g["high"], g["low"], g["close"], period=14).values
        g["vwap"]     = calc_vwap(g["high"], g["low"], g["close"],
                                  g["volume_quote"], g["timestamp"]).values
        g["vwap_dev"] = (g["close"] - g["vwap"]) / g["vwap"]

        bb = calc_bollinger(g["close"], period=20)
        g  = pd.concat([g, bb], axis=1)

        g["zscore_20"]   = calc_zscore(g["close"], period=20).values
        g["zscore_60"]   = calc_zscore(g["close"], period=60).values
        g["hurst_100"]   = calc_hurst(g["close"], window=100).values
        g["halflife_100"]= calc_halflife(g["close"], window=100).values

        results.append(g)
        print(f"  [OK] {ticker}: {len(g)} Kerzen")

    return pd.concat(results, ignore_index=True)

df_features = compute_indicators(df_raw)
print(f"\nFertig: {len(df_features):,} Zeilen, {len(df_features.columns)} Spalten")
```

### Zeile für Zeile:

**`for ticker, group in df.groupby("market_symbol", sort=False):`**
- Teilt den gesamten DataFrame in Gruppen — eine pro Ticker.
- `sort=False` behält die ursprüngliche Reihenfolge bei (schneller als sortieren).
- `ticker` = z.B. "BTCUSDT", `group` = alle Zeilen dieses Tickers.

**`g = group.copy().sort_values("timestamp").reset_index(drop=True)`**
- `.copy()` → Erstellt eine echte Kopie, damit Änderungen an `g` nicht
  den originalen DataFrame `df_raw` verändern.
- `.sort_values("timestamp")` → Chronologisch sortieren (Sicherheitsschritt).
- `.reset_index(drop=True)` → Sauberer Index 0, 1, 2, ...

**Warum `.values` bei jeder Zuweisung?**
```python
g["rsi_7"] = calc_rsi(g["close"], period=7).values
```
- `calc_rsi(...)` gibt eine Pandas Series mit eigenem Index zurück.
- Dieser Index könnte von `g`'s Index abweichen (nach reset_index).
- `.values` extrahiert ein rohes NumPy-Array **ohne Index**.
- Ohne `.values` könnte Pandas versuchen, die Werte nach Index auszurichten,
  was zu NaN-Werten oder falschen Zuordnungen führen würde.

**`bb = calc_bollinger(g["close"], period=20)` und `g = pd.concat([g, bb], axis=1)`**
- Bollinger Bands gibt einen ganzen DataFrame zurück (4 Spalten).
- `pd.concat([g, bb], axis=1)` fügt diese Spalten seitlich an `g` an.
- Hier wird kein `.values` verwendet, weil `pd.concat` die Indizes automatisch ausrichtet.

**`results.append(g)`**
- Sammelt alle fertigen Ticker-DataFrames in einer Liste.

**`return pd.concat(results, ignore_index=True)`**
- Fügt alle Ticker wieder zu einem einzigen DataFrame zusammen.
- `ignore_index=True` → Durchgehender Index über alle Ticker hinweg.

---

## 11. Speichern

```python
out = FEATURES_DIR / "ohlcv_features.parquet"
df_features.to_parquet(out, index=False)
print(f"Gespeichert: {out}  ({out.stat().st_size / 1_048_576:.1f} MB)")
```

- **Parquet-Format**: Spaltenorientiert, komprimiert, schnell lesbar.
  Ideal für grosse analytische Datensätze.
- `index=False` → Der DataFrame-Index wird nicht mitgespeichert (spart Platz).
- `out.stat().st_size / 1_048_576` → Dateigrösse in Megabytes
  (1'048'576 = 1024 × 1024 = 1 MB in Bytes).

---

## 12. Schnellcheck

```python
SAMPLE_TICKER = "BTCUSDT"
sample = df_features[df_features["market_symbol"] == SAMPLE_TICKER].tail(20)

cols = ["timestamp", "close", "rsi_14", "bb_pct_20",
        "zscore_20", "vwap_dev", "atr_14", "hurst_100", "halflife_100"]

display_df = sample[cols].copy()
display_df["timestamp"]    = display_df["timestamp"].dt.strftime("%Y-%m-%d %H:%M")
display_df["close"]        = display_df["close"].map("${:,.2f}".format)
display_df["rsi_14"]       = display_df["rsi_14"].map("{:.1f}".format)
display_df["bb_pct_20"]    = display_df["bb_pct_20"].map("{:.3f}".format)
display_df["zscore_20"]    = display_df["zscore_20"].map("{:.3f}".format)
display_df["vwap_dev"]     = display_df["vwap_dev"].map("{:.4f}".format)
display_df["atr_14"]       = display_df["atr_14"].map("{:.2f}".format)
display_df["hurst_100"]    = display_df["hurst_100"].map(lambda x: f"{x:.3f}" if pd.notna(x) else "–")
display_df["halflife_100"] = display_df["halflife_100"].map(lambda x: f"{x:.1f}" if pd.notna(x) else "–")

print(f"\n{SAMPLE_TICKER} — letzte 20 Kerzen:")
print(display_df.to_string(index=False))
```

### Was hier passiert:

**`df_features[df_features["market_symbol"] == SAMPLE_TICKER].tail(20)`**
- Filtert nur BTC-Daten und nimmt die letzten 20 Kerzen (die neuesten).

**Formatierungs-Zeilen:**
- `.dt.strftime("%Y-%m-%d %H:%M")` → Timestamp in lesbares Format: `2024-06-15 14:30`
- `"${:,.2f}".format` → Dollar-Format: `$67,234.50`
- `"{:.1f}".format` → Eine Dezimalstelle: `45.3`
- `"{:.3f}".format` → Drei Dezimalstellen: `0.734`
- `lambda x: f"{x:.3f}" if pd.notna(x) else "–"` → Zeigt `–` wenn der Wert NaN ist
  (bei Hurst/Half-Life möglich, wenn nicht genug Daten vorliegen).

**`display_df.to_string(index=False)`**
- Gibt den DataFrame als formatierten Text aus, ohne den Index links.

---

## Zusammenfassung: Alle Features und ihre Rollen

| Feature | Funktion | Eingabe | Ausgabe-Spalte(n) |
|---|---|---|---|
| **RSI** | Momentum-Oszillator (0–100) | close | `rsi_7`, `rsi_14` |
| **ATR** | Durchschnittliche Volatilität | high, low, close | `atr_14` |
| **VWAP** | Volumengewichteter Tagespreis | high, low, close, volume, timestamp | `vwap`, `vwap_dev` |
| **Bollinger Bands** | Volatilitätskanal | close | `bb_mid_20`, `bb_upper_20`, `bb_lower_20`, `bb_pct_20` |
| **Z-Score** | Statist. Abweichung vom Mittelwert | close | `zscore_20`, `zscore_60` |
| **Hurst-Exponent** | Regime-Erkennung (Mean-Rev. vs Trend) | close | `hurst_100` |
| **Half-Life** | Geschwindigkeit der Mean-Reversion | close | `halflife_100` |

### Wie die Features zusammenspielen:

```
1. FILTER:    Hurst < 0.5?           → Ja: Markt ist mean-reverting, weiter
                                      → Nein: Nicht handeln

2. SIGNAL:    Z-Score ≤ −2?          → Long-Signal (Preis zu tief)
              Z-Score ≥ +2?          → Short-Signal (Preis zu hoch)
              BB_pct < 0 oder > 1?   → Bestätigung durch Bollinger
              RSI < 30 oder > 70?    → Bestätigung durch RSI

3. SIZING:    ATR bestimmt Stop-Loss → daraus Positionsgrösse

4. EXIT:      Half-Life bestimmt Time-Stop
              VWAP_dev ≈ 0 → Preis zurück beim Fair Value → Position schliessen
```

In [ ]:
PLOTS_DIR    = PROJECT_ROOT / "data" / "processed" / "plots"


# Plots-Ordner leeren und neu erstellen
if PLOTS_DIR.exists():
    shutil.rmtree(PLOTS_DIR)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

# ─── Daten laden ───────────────────────────────────────────────
df = pd.read_parquet(FEATURES_DIR / "ohlcv_features.parquet")
df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)
df["date"]      = df["timestamp"].dt.date          # z.B. 2026-01-15


# ─── Plot-Funktion für einen Tag + einen Ticker ───────────────
def plot_ticker_day(df_td: pd.DataFrame, ticker: str, day_label: str) -> plt.Figure:
    ts = df_td["timestamp"]

    # Start- und Endkurs des Tages
    close_start = df_td["close"].iloc[0]
    close_end   = df_td["close"].iloc[-1]
    pct_change  = (close_end - close_start) / close_start * 100
    color_pct   = "#66BB6A" if pct_change >= 0 else "#EF5350"
    arrow_pct   = "▲" if pct_change >= 0 else "▼"

    fig = plt.figure(figsize=(18, 22))
    fig.suptitle(
        f"{ticker}  —  {day_label}    "
        f"Open: {close_start:,.2f}  →  Close: {close_end:,.2f}  "
        f"({arrow_pct} {pct_change:+.2f}%)",
        fontsize=14, fontweight="bold", y=0.98,
        color=color_pct
    )
    gs = gridspec.GridSpec(6, 1, hspace=0.45)

    # ── Panel 1: Preis + BB + VWAP + Close-Beschriftungen ──
    ax1 = fig.add_subplot(gs[0])
    ax1.plot(ts, df_td["close"],        color="#2196F3", lw=1.2, label="Close")
    ax1.plot(ts, df_td["bb_mid_20"],    color="gray",   lw=0.8, ls="--", label="BB Mid")
    ax1.plot(ts, df_td["bb_upper_20"],  color="#EF5350", lw=0.8, ls="--", label="BB Upper")
    ax1.plot(ts, df_td["bb_lower_20"],  color="#66BB6A", lw=0.8, ls="--", label="BB Lower")
    ax1.plot(ts, df_td["vwap"],         color="#FF9800", lw=1.0, ls="-.", label="VWAP")
    ax1.fill_between(ts, df_td["bb_lower_20"], df_td["bb_upper_20"], alpha=0.06, color="#2196F3")

    # Close-Beschriftungen: Start, Ende, Min, Max des Tages
    close_min_idx = df_td["close"].idxmin()
    close_max_idx = df_td["close"].idxmax()

    for idx, label, va, color in [
        (df_td.index[0],  f"Open {close_start:,.2f}",                      "bottom", "#2196F3"),
        (df_td.index[-1], f"Close {close_end:,.2f}",                       "bottom", color_pct),
        (close_min_idx,   f"Min {df_td['close'].min():,.2f}",              "top",    "#EF5350"),
        (close_max_idx,   f"Max {df_td['close'].max():,.2f}",              "bottom", "#66BB6A"),
    ]:
        ax1.annotate(
            label,
            xy=(df_td.loc[idx, "timestamp"], df_td.loc[idx, "close"]),
            fontsize=7, fontweight="bold", color=color,
            textcoords="offset points", xytext=(0, -12 if va == "top" else 12),
            ha="center", va=va,
            arrowprops=dict(arrowstyle="-", color=color, lw=0.5),
        )

    ax1.set_ylabel("Preis (USDT)")
    ax1.legend(loc="upper left", fontsize=7, ncol=5)
    ax1.set_title("Preis + Bollinger Bands + VWAP", fontsize=9)

    # ── Panel 2: RSI ──
    ax2 = fig.add_subplot(gs[1], sharex=ax1)
    ax2.plot(ts, df_td["rsi_7"],  color="#AB47BC", lw=1.0, label="RSI 7")
    ax2.plot(ts, df_td["rsi_14"], color="#7E57C2", lw=1.0, label="RSI 14")
    ax2.axhline(70, color="#EF5350", lw=0.8, ls="--")
    ax2.axhline(30, color="#66BB6A", lw=0.8, ls="--")
    ax2.axhline(50, color="gray",    lw=0.5, ls=":")
    ax2.fill_between(ts, 70, 100, alpha=0.07, color="#EF5350")
    ax2.fill_between(ts, 0,  30,  alpha=0.07, color="#66BB6A")
    ax2.set_ylim(0, 100)
    ax2.set_ylabel("RSI")
    ax2.legend(loc="upper left", fontsize=7, ncol=2)
    ax2.set_title("RSI — überkauft >70 / überverkauft <30", fontsize=9)

    # ── Panel 3: Z-Score ──
    ax3 = fig.add_subplot(gs[2], sharex=ax1)
    ax3.plot(ts, df_td["zscore_20"], color="#26A69A", lw=1.0, label="Z-Score 20")
    ax3.plot(ts, df_td["zscore_60"], color="#00796B", lw=1.0, label="Z-Score 60", alpha=0.7)
    ax3.axhline( 2, color="#EF5350", lw=0.8, ls="--")
    ax3.axhline(-2, color="#66BB6A", lw=0.8, ls="--")
    ax3.axhline( 0, color="gray",    lw=0.5, ls=":")
    ax3.fill_between(ts, df_td["zscore_20"], 0, where=df_td["zscore_20"] > 0, alpha=0.08, color="#EF5350")
    ax3.fill_between(ts, df_td["zscore_20"], 0, where=df_td["zscore_20"] < 0, alpha=0.08, color="#66BB6A")
    ax3.set_ylabel("Z-Score")
    ax3.legend(loc="upper left", fontsize=7, ncol=2)
    ax3.set_title("Z-Score — Signal bei |z| > 2", fontsize=9)

    # ── Panel 4: ATR ──
    ax4 = fig.add_subplot(gs[3], sharex=ax1)
    ax4.plot(ts, df_td["atr_14"], color="#FF7043", lw=1.0)
    ax4.fill_between(ts, 0, df_td["atr_14"], alpha=0.2, color="#FF7043")
    ax4.set_ylabel("ATR 14")
    ax4.set_title("ATR — Volatilität (höher = unsicherer Markt)", fontsize=9)

    # ── Panel 5: Hurst ──
    ax5 = fig.add_subplot(gs[4], sharex=ax1)
    ax5.plot(ts, df_td["hurst_100"], color="#5C6BC0", lw=1.0)
    ax5.axhline(0.5, color="gray", lw=0.8, ls="--", label="H=0.5 (random walk)")
    ax5.fill_between(ts, df_td["hurst_100"], 0.5, where=df_td["hurst_100"] < 0.5,
                     alpha=0.15, color="#66BB6A", label="mean-reverting")
    ax5.fill_between(ts, df_td["hurst_100"], 0.5, where=df_td["hurst_100"] > 0.5,
                     alpha=0.15, color="#EF5350", label="trending")
    ax5.set_ylabel("Hurst H")
    ax5.legend(loc="upper left", fontsize=7, ncol=3)
    ax5.set_title("Hurst-Exponent — grün = mean-reverting (H<0.5)", fontsize=9)

    # ── Achsen formatieren ──
    for ax in [ax1, ax2, ax3, ax4, ax5]:
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
        ax.xaxis.set_major_locator(mdates.HourLocator(interval=2))
        plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha="right", fontsize=7)
        ax.grid(True, alpha=0.2)

    return fig


# 20 zufällige Tage plotten

combos = [
    (ticker, date)
    for ticker in df["market_symbol"].unique()
    for date, df_td in df[df["market_symbol"] == ticker].groupby("date")
    if len(df_td) >= 20
]

random.seed(42)
sample = random.sample(combos, min(20, len(combos)))
log.info(f"Verfügbare Kombinationen: {len(combos)} → Generiere {len(sample)} Plots\n")

matplotlib.use("Agg")

for i, (ticker, date) in enumerate(sample, 1):
    df_td     = df[(df["market_symbol"] == ticker) & (df["date"] == date)]
    day_label = str(date)
    fig       = plot_ticker_day(df_td, ticker, day_label)
    out       = PLOTS_DIR / f"{ticker}_{day_label}.png"
    fig.savefig(out, dpi=100, bbox_inches="tight")
    plt.close(fig)
    plt.close("all")
    gc.collect()
    log.info(f"  [{i}/{len(sample)}] {out.name}")

log.info(f"\nAlle Plots gespeichert in: {PLOTS_DIR}")